# No.3 2次元フーリエ変換 — 全課題パイプライン

1. mri128def.mat に2D FFT → NMR信号(mat)作成
2. NMR信号をIFFT → pgm画像で再構成像出力
3. extract: 中心部/周辺部をd=25,50,100で抽出
4. 抽出後データをIFFT → pgm画像で再構成像出力（6パターン）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
import scipy.io
import skimage.io
import os

os.makedirs("output", exist_ok=True)
imgSize = 128

## 課題1: mri128def.mat に2D FFT → NMR信号作成

In [ ]:
data = scipy.io.loadmat("mri128def.mat")
keys = [k for k in data.keys() if not k.startswith("_")]
Signal_orig = data[keys[0]].astype(complex)
print(f"入力: {keys[0]}, shape={Signal_orig.shape}")

# 2D FFT
Signal_FFT = np.fft.fftshift(np.fft.fft2(np.fft.fftshift(Signal_orig)))
scipy.io.savemat("mri128def_FFT2D.mat", {"Signal": Signal_FFT})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(np.abs(Signal_orig), cmap="gray")
axes[0].set_title("mri128def (入力画像)")
axes[0].axis("off")
axes[1].imshow(np.log1p(np.abs(Signal_FFT)), cmap="gray")
axes[1].set_title("2D FFT (log magnitude)")
axes[1].axis("off")
fig.tight_layout()
fig.savefig("output/kadai1_FFT2D.png", dpi=150)
plt.show()

## 課題2: NMR信号をIFFTして画像再構成（pgm出力）

In [ ]:
def to_pgm(signal, filename):
    """複素信号を絶対値→正規化→uint8でpgm保存"""
    img = np.abs(signal)
    if img.max() > 0:
        img = img / img.max() * 255
    img_u8 = img.astype(np.uint8)
    skimage.io.imsave(filename, img_u8)
    return img_u8

# IFFT再構成
recon = np.fft.fftshift(np.fft.ifft2(np.fft.fftshift(Signal_FFT)))
recon_img = to_pgm(recon, "output/mri128def_recon.pgm")

plt.figure(figsize=(6, 6))
plt.imshow(recon_img, cmap="gray")
plt.title("再構成像 (IFFT)")
plt.axis("off")
plt.savefig("output/kadai2_recon.png", dpi=150, bbox_inches="tight")
plt.show()
print("mri128def_recon.pgm を保存しました")

## 課題4: extract → IFFT → pgm再構成（d=25,50,100 × 中心C/周辺A = 6パターン）

In [ ]:
d_list = [25, 50, 100]

for flag, flag_label in [("C", "中心部抽出"), ("A", "周辺部抽出")]:
    fig, axes = plt.subplots(1, len(d_list), figsize=(15, 5))
    fig.suptitle(f"{flag_label} ({flag}) → IFFT再構成", fontsize=14)

    for idx, d in enumerate(d_list):
        # マスク作成
        border = imgSize // 2 - d // 2
        mask = np.ones((imgSize, imgSize))
        if flag == "C":
            mask[:] = 0
            mask[border:border + d, border:border + d] = 1
        else:
            mask[border:border + d, border:border + d] = 0

        # マスク適用 → IFFT
        Signal_masked = Signal_FFT * mask
        out_name = f"mri128defFFT2D{flag}{d}"
        scipy.io.savemat(f"output/{out_name}.mat", {"Signal": Signal_masked})

        recon = np.fft.fftshift(np.fft.ifft2(np.fft.fftshift(Signal_masked)))
        recon_img = to_pgm(recon, f"output/{out_name}_recon.pgm")

        axes[idx].imshow(recon_img, cmap="gray")
        axes[idx].set_title(f"d={d}")
        axes[idx].axis("off")

    fig.tight_layout()
    fig.savefig(f"output/kadai4_{flag}_recon.png", dpi=150)
    plt.show()

print("全6パターンの再構成像を保存しました")